# Chapitre 5 — Nettoyage des données

**Durée estimée : 10-12 heures**

---

## Objectifs d'apprentissage

À la fin de ce chapitre, vous serez capable de :

1. **Appliquer** différentes stratégies de traitement des valeurs manquantes (suppression, imputation)
2. **Identifier et supprimer** les doublons en préservant les informations pertinentes
3. **Traiter** les valeurs aberrantes selon le contexte métier
4. **Nettoyer** les types de données (dates, numériques, texte) pour les rendre exploitables

---

## 🎯 Le Hook : La startup qui a perdu 2 millions à cause de doublons

En 2021, une startup e-commerce française a lancé une campagne marketing ciblée. Ils ont envoyé un code promo de 50€ à leurs "100 000 meilleurs clients". Le problème ? Leur base contenait **40% de doublons**.

Résultat : 40 000 personnes ont reçu plusieurs codes. Les plus malins ont cumulé les réductions. Perte totale : **2 millions d'euros**.

Le nettoyage des données n'est pas un luxe académique. C'est une **nécessité business**.

> 💭 **Question Socratique #1** : Cette erreur aurait-elle pu être évitée par une simple requête SQL de dé-duplication ? Ou le problème était-il plus profond (processus, culture, outils) ?

---

# 📖 PARTIE THÉORIQUE

---

## 5.1 Gestion des valeurs manquantes

### Les trois stratégies principales

```
┌─────────────────────────────────────────────────────────────────────┐
│              STRATÉGIES DE TRAITEMENT DES MISSING                   │
├─────────────────────┬─────────────────────┬─────────────────────────┤
│     SUPPRESSION     │     IMPUTATION      │        FLAG             │
│                     │                     │                         │
│  Supprimer lignes   │  Remplacer par      │  Créer une colonne      │
│  ou colonnes        │  une valeur         │  indicatrice            │
│                     │  estimée            │                         │
├─────────────────────┼─────────────────────┼─────────────────────────┤
│  • dropna()         │  • fillna(valeur)   │  • isna().astype(int)   │
│  • Simple mais      │  • Moyenne/médiane  │  • Préserve             │
│    perte de données │  • Par groupe       │    l'information        │
└─────────────────────┴─────────────────────┴─────────────────────────┘
```

### Stratégie 1 : Suppression — Quand l'utiliser ?

| Situation | Action recommandée |
|-----------|-------------------|
| < 5% de missing dans une colonne | Supprimer les lignes concernées |
| > 50% de missing dans une colonne | Supprimer la colonne entière |
| Données MCAR (manque aléatoire) | Suppression acceptable |
| Donnée critique pour l'analyse | Ne pas supprimer, imputer |

---

# 🖥️ PARTIE PRATIQUE

---

## 5.1 Gestion des valeurs manquantes (Pratique)

### Suppression avec pandas

In [ ]:
import pandas as pd
import numpy as np

# Créer un DataFrame de démonstration
np.random.seed(42)
df = pd.DataFrame({
    'id': range(100),
    'nom': [f'Client_{i}' if np.random.random() > 0.05 else None for i in range(100)],
    'email': [f'email_{i}@test.com' if np.random.random() > 0.02 else None for i in range(100)],
    'age': [np.random.randint(18, 70) if np.random.random() > 0.08 else None for i in range(100)],
    'revenu': [np.random.randint(20000, 100000) if np.random.random() > 0.25 else None for i in range(100)],
    'notes': [f'Note_{i}' if np.random.random() > 0.80 else None for i in range(100)]
})

print("DataFrame avec valeurs manquantes :")
print(f"Shape : {df.shape}")
print("\nPourcentage de missing par colonne :")
print((df.isnull().mean() * 100).round(1))

In [ ]:
# Supprimer les lignes avec au moins une valeur manquante
df_dropna_all = df.dropna()
print(f"Après dropna() : {len(df_dropna_all)} lignes (supprimé {len(df) - len(df_dropna_all)})")

In [ ]:
# Supprimer les lignes où une colonne spécifique est manquante
df_dropna_subset = df.dropna(subset=['email', 'nom'])
print(f"Après dropna(subset=['email', 'nom']) : {len(df_dropna_subset)} lignes")

In [ ]:
# Supprimer les colonnes avec plus de 50% de missing
seuil = len(df) * 0.5
df_dropna_cols = df.dropna(axis=1, thresh=seuil)
print(f"\nColonnes après suppression (>50% missing) : {list(df_dropna_cols.columns)}")
print(f"Colonnes supprimées : {set(df.columns) - set(df_dropna_cols.columns)}")

In [ ]:
# Supprimer uniquement si TOUTES les valeurs sont manquantes
df_dropna_how = df.dropna(how='all')
print(f"Après dropna(how='all') : {len(df_dropna_how)} lignes")

### ✍️ Exercice 5.1 : Décision de suppression (10 min)

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
df_ex = pd.DataFrame({
    'id': range(1000),
    'nom': ['Client_' + str(i) for i in range(1000)],
    'age': np.where(np.random.random(1000) < 0.03, np.nan, np.random.randint(18, 70, 1000)),
    'email': np.where(np.random.random(1000) < 0.02, np.nan, ['email_' + str(i) + '@test.com' for i in range(1000)]),
    'revenu': np.where(np.random.random(1000) < 0.25, np.nan, np.random.randint(20000, 100000, 1000)),
    'notes_internes': np.where(np.random.random(1000) < 0.80, np.nan, ['Note_' + str(i) for i in range(1000)])
})

# Analyse
print("Pourcentage de missing par colonne :")
print((df_ex.isnull().mean() * 100).round(1))

In [ ]:
# Décisions :
print("\n=== DÉCISIONS ===")
print("- id : 0% → Pas d'action nécessaire")
print("- nom : 0% → Pas d'action nécessaire")
print("- age (~3% missing) : Supprimer les lignes OU imputer par médiane")
print("- email (~2% missing) : Supprimer les lignes (email souvent critique)")
print("- revenu (~25% missing) : Imputer (trop de données à perdre)")
print("- notes_internes (~80% missing) : Supprimer la colonne entière")

### Stratégie 2 : Imputation

In [ ]:
# Recréer le DataFrame pour les exemples d'imputation
np.random.seed(42)
df = pd.DataFrame({
    'categorie': np.random.choice(['A', 'B', 'C', None], 100, p=[0.3, 0.3, 0.3, 0.1]),
    'age': np.where(np.random.random(100) < 0.1, np.nan, np.random.randint(20, 60, 100)),
    'revenu': np.where(np.random.random(100) < 0.15, np.nan, np.random.randint(25000, 80000, 100)),
    'ville': np.random.choice(['Paris', 'Lyon', 'Marseille', None], 100, p=[0.3, 0.3, 0.3, 0.1])
})

print("Avant imputation :")
print(df.isnull().sum())

In [ ]:
# Imputation par valeur fixe (catégorielles)
df['categorie'] = df['categorie'].fillna('Inconnu')
print("Catégorie après fillna('Inconnu') :")
print(df['categorie'].value_counts())

In [ ]:
# Imputation par moyenne (numériques)
moyenne_age = df['age'].mean()
df['age'] = df['age'].fillna(moyenne_age)
print(f"\nÂge imputé par moyenne : {moyenne_age:.1f}")
print(f"Valeurs manquantes restantes : {df['age'].isnull().sum()}")

In [ ]:
# Imputation par médiane (plus robuste aux outliers)
mediane_revenu = df['revenu'].median()
df['revenu'] = df['revenu'].fillna(mediane_revenu)
print(f"\nRevenu imputé par médiane : {mediane_revenu:.0f}")
print(f"Valeurs manquantes restantes : {df['revenu'].isnull().sum()}")

In [ ]:
# Imputation par mode (catégorielles)
mode_ville = df['ville'].mode()[0]
df['ville'] = df['ville'].fillna(mode_ville)
print(f"\nVille imputée par mode : {mode_ville}")
print(f"Valeurs manquantes restantes : {df['ville'].isnull().sum()}")

### Imputation contextuelle (par groupe)

In [ ]:
# Créer un DataFrame avec groupes
np.random.seed(42)
df_groupe = pd.DataFrame({
    'departement': np.random.choice(['Paris', 'Lyon', 'Marseille'], 100),
    'salaire': np.where(np.random.random(100) < 0.2, np.nan, np.random.randint(30000, 80000, 100))
})

# Ajuster les salaires par département
ajustements = {'Paris': 1.2, 'Lyon': 1.0, 'Marseille': 0.9}
for dept, mult in ajustements.items():
    mask = (df_groupe['departement'] == dept) & df_groupe['salaire'].notna()
    df_groupe.loc[mask, 'salaire'] = df_groupe.loc[mask, 'salaire'] * mult

print("Avant imputation par groupe :")
print(df_groupe.groupby('departement')['salaire'].agg(['count', 'median']))

In [ ]:
# Imputer le salaire par la médiane du département
df_groupe['salaire'] = df_groupe.groupby('departement')['salaire'].transform(
    lambda x: x.fillna(x.median())
)

print("\nAprès imputation par groupe :")
print(df_groupe.groupby('departement')['salaire'].agg(['count', 'median']))
print(f"\nValeurs manquantes restantes : {df_groupe['salaire'].isnull().sum()}")

### Forward/Backward fill (séries temporelles)

In [ ]:
# Données temporelles avec missing
df_temp = pd.DataFrame({
    'date': pd.date_range('2024-01-01', periods=10),
    'temperature': [15, 16, np.nan, np.nan, 18, 19, np.nan, 21, 22, 23]
})

print("Données originales :")
print(df_temp)

In [ ]:
# Forward fill (propager la dernière valeur connue)
df_temp['temp_ffill'] = df_temp['temperature'].ffill()

# Backward fill (propager la prochaine valeur connue)
df_temp['temp_bfill'] = df_temp['temperature'].bfill()

# Interpolation linéaire
df_temp['temp_interp'] = df_temp['temperature'].interpolate(method='linear')

print("Comparaison des méthodes :")
print(df_temp)

### Stratégie 3 : Flag (indicateur)

Parfois, le fait qu'une donnée soit manquante **est une information en soi**.

In [ ]:
# Créer un DataFrame
np.random.seed(42)
df_flag = pd.DataFrame({
    'client_id': range(10),
    'revenu': [50000, np.nan, 45000, 60000, np.nan, np.nan, 55000, 48000, np.nan, 52000]
})

print("Avant création du flag :")
print(df_flag)

In [ ]:
# Créer une colonne indicatrice AVANT imputation
df_flag['revenu_manquant'] = df_flag['revenu'].isna().astype(int)

# Puis imputer le revenu
df_flag['revenu'] = df_flag['revenu'].fillna(df_flag['revenu'].median())

print("Après flag + imputation :")
print(df_flag)

# Maintenant vous avez les deux informations :
# - revenu imputé
# - indicateur si c'était manquant

### ✍️ Exercice 5.2 : Imputation complète (15 min)

In [ ]:
import pandas as pd
import numpy as np

# Données avec patterns réalistes
np.random.seed(42)
df_ex2 = pd.DataFrame({
    'client_id': range(500),
    'age': np.where(np.random.random(500) < 0.1, np.nan, np.random.randint(20, 70, 500).astype(float)),
    'departement': np.random.choice(['Paris', 'Lyon', 'Marseille', 'Bordeaux'], 500),
    'salaire': np.where(np.random.random(500) < 0.2, np.nan, np.random.randint(25000, 80000, 500).astype(float)),
    'anciennete_mois': np.where(np.random.random(500) < 0.05, np.nan, np.random.randint(1, 120, 500).astype(float))
})

# Ajuster les salaires par département
salaire_moyen = {'Paris': 55000, 'Lyon': 45000, 'Marseille': 42000, 'Bordeaux': 43000}
for dept, moyenne in salaire_moyen.items():
    mask = (df_ex2['departement'] == dept) & df_ex2['salaire'].notna()
    df_ex2.loc[mask, 'salaire'] = df_ex2.loc[mask, 'salaire'] * (moyenne / 50000)

print("Avant nettoyage :")
print(df_ex2.isnull().sum())

In [ ]:
# 1. Créer un flag pour les salaires manquants (AVANT imputation)
df_ex2['salaire_manquant'] = df_ex2['salaire'].isna().astype(int)

# 2. Imputer l'âge par la médiane globale
df_ex2['age'] = df_ex2['age'].fillna(df_ex2['age'].median())

# 3. Imputer le salaire par la médiane du département
df_ex2['salaire'] = df_ex2.groupby('departement')['salaire'].transform(
    lambda x: x.fillna(x.median())
)

# 4. Supprimer les lignes où anciennete_mois est manquant (< 5%)
df_ex2 = df_ex2.dropna(subset=['anciennete_mois'])

print("\nAprès nettoyage :")
print(df_ex2.isnull().sum())
print(f"\nLignes restantes : {len(df_ex2)}")
print(f"Salaires qui étaient manquants : {df_ex2['salaire_manquant'].sum()}")

> 💭 **Question Socratique #2** : Imputer les revenus par la moyenne peut-il créer un biais si les revenus manquants ne sont pas aléatoires ? Par exemple, si les hauts revenus refusent plus souvent de déclarer ?

---

## 5.2 Traitement des doublons

### Identification complète

In [ ]:
# Créer un DataFrame avec des doublons
df_doublons = pd.DataFrame({
    'id': [1, 2, 3, 4, 5, 1, 2],
    'nom': ['Alice', 'Bob', 'Charlie', 'David', 'Eve', 'Alice', 'Bob'],
    'email': ['a@t.com', 'b@t.com', 'c@t.com', 'd@t.com', 'e@t.com', 'a@t.com', 'b@t.com'],
    'montant': [100, 200, 150, 300, 250, 100, 200]
})

print("DataFrame avec doublons :")
print(df_doublons)

In [ ]:
# Nombre de doublons exacts
print(f"Doublons exacts : {df_doublons.duplicated().sum()}")

# Voir toutes les lignes dupliquées (y compris les originaux)
doublons = df_doublons[df_doublons.duplicated(keep=False)]
print("\nToutes les lignes dupliquées :")
print(doublons.sort_values(by=['id']))

In [ ]:
# Doublons sur une clé spécifique
doublons_email = df_doublons[df_doublons.duplicated(subset=['email'], keep=False)]
print(f"Emails en double : {len(doublons_email)}")
print(doublons_email)

### Suppression des doublons

In [ ]:
# Supprimer les doublons exacts (garder la première occurrence)
df_clean = df_doublons.drop_duplicates()
print("Après drop_duplicates() (keep='first' par défaut) :")
print(df_clean)

In [ ]:
# Garder la dernière occurrence
df_clean_last = df_doublons.drop_duplicates(keep='last')
print("Après drop_duplicates(keep='last') :")
print(df_clean_last)

In [ ]:
# Dé-dupliquer sur des colonnes spécifiques
df_clean_email = df_doublons.drop_duplicates(subset=['email'])
print("Après drop_duplicates(subset=['email']) :")
print(df_clean_email)

In [ ]:
# Dé-dupliquer en gardant la ligne avec le montant le plus élevé
df_doublons_v2 = pd.DataFrame({
    'client_id': [1, 2, 3, 1, 2],
    'nom': ['Alice', 'Bob', 'Charlie', 'Alice', 'Bob'],
    'montant': [100, 200, 150, 500, 50]  # Montants différents
})

print("Données avec montants différents :")
print(df_doublons_v2)

df_clean_max = df_doublons_v2.sort_values('montant', ascending=False).drop_duplicates(subset=['client_id'])
print("\nGarder le montant le plus élevé par client :")
print(df_clean_max)

### Gestion des quasi-doublons

In [ ]:
# Quasi-doublons : même entité avec des variations mineures
df_quasi = pd.DataFrame({
    'id': [1, 2, 3, 4],
    'nom': ['Alice Martin', 'ALICE MARTIN', 'Bob Dupont', 'bob dupont'],
    'email': ['alice@test.com', 'ALICE@TEST.COM', 'bob@test.com', 'bob@test.com'],
    'date_maj': ['2024-01-01', '2024-03-15', '2024-02-01', '2024-04-10']
})

print("Données avec quasi-doublons :")
print(df_quasi)

In [ ]:
# Normaliser avant de comparer
df_quasi['nom_normalise'] = df_quasi['nom'].str.lower().str.strip()
df_quasi['email_normalise'] = df_quasi['email'].str.lower().str.strip()

# Détecter les quasi-doublons
quasi_doublons = df_quasi[df_quasi.duplicated(subset=['nom_normalise', 'email_normalise'], keep=False)]
print(f"\nQuasi-doublons détectés : {len(quasi_doublons)}")
print(quasi_doublons)

In [ ]:
# Fusionner les quasi-doublons (garder le plus récent)
df_clean_quasi = df_quasi.sort_values('date_maj', ascending=False)
df_clean_quasi = df_clean_quasi.drop_duplicates(subset=['nom_normalise', 'email_normalise'], keep='first')
df_clean_quasi = df_clean_quasi.drop(columns=['nom_normalise', 'email_normalise'])

print("\nAprès fusion des quasi-doublons :")
print(df_clean_quasi)

### ✍️ Exercice 5.3 : Dé-duplication intelligente (15 min)

In [ ]:
import pandas as pd

# Données avec différents types de doublons
df_ex3 = pd.DataFrame({
    'id': [1, 2, 3, 4, 5, 6, 7, 8],
    'nom': ['Alice Martin', 'Bob Dupont', 'alice martin', 'Charlie Brown',
            'Bob Dupont', 'David Lee', 'ALICE MARTIN', 'Eve Wilson'],
    'email': ['alice@test.com', 'bob@test.com', 'alice@test.com', 'charlie@test.com',
              'bob@test.com', 'david@test.com', 'alice@test.com', 'eve@test.com'],
    'date_inscription': ['2024-01-15', '2024-01-16', '2024-02-01', '2024-01-17',
                         '2024-03-01', '2024-01-18', '2024-03-15', '2024-01-19'],
    'montant_total': [500, 1200, 300, 800, 1500, 600, 200, 900]
})

print("Données originales :")
print(df_ex3)

In [ ]:
# Étape 1 : Normaliser le nom et l'email
df_ex3['nom_norm'] = df_ex3['nom'].str.lower().str.strip()
df_ex3['email_norm'] = df_ex3['email'].str.lower().str.strip()

# Étape 2 : Identifier les doublons sur nom_norm + email_norm
doublons_ex3 = df_ex3[df_ex3.duplicated(subset=['nom_norm', 'email_norm'], keep=False)]
print(f"\nDoublons identifiés : {len(doublons_ex3)}")
print(doublons_ex3[['id', 'nom', 'email', 'montant_total']])

In [ ]:
# Étape 3 : Garder l'enregistrement avec le montant_total le plus élevé
df_clean_ex3 = df_ex3.sort_values('montant_total', ascending=False)
df_clean_ex3 = df_clean_ex3.drop_duplicates(subset=['nom_norm', 'email_norm'], keep='first')

# Étape 4 : Nettoyer les colonnes temporaires
df_clean_ex3 = df_clean_ex3.drop(columns=['nom_norm', 'email_norm'])

print(f"\nRésultat : {len(df_clean_ex3)} lignes uniques")
print(df_clean_ex3)

# Question : Pourquoi avons-nous gardé la ligne avec le montant le plus élevé ?
print("\n→ On garde le client avec le plus de valeur (meilleur client)")

---

## 5.3 Traitement des valeurs aberrantes

### Les trois approches

| Approche | Description | Quand l'utiliser |
|----------|-------------|------------------|
| **Suppression** | Retirer les outliers | Erreurs évidentes (âge négatif) |
| **Winsorisation** | Ramener aux bornes | Conserver toutes les lignes |
| **Conservation** | Ne rien faire | Outlier = information réelle |

### Suppression conditionnelle

In [ ]:
# Données avec outliers
np.random.seed(42)
df_outliers = pd.DataFrame({
    'id': range(100),
    'age': np.concatenate([np.random.randint(20, 65, 97), [-5, 150, 200]]),
    'salaire': np.concatenate([np.random.normal(50000, 10000, 96), [500000, -1000, 48000, 52000]])
})

print("Statistiques avant nettoyage :")
print(df_outliers.describe())

In [ ]:
# Supprimer les valeurs impossibles (âge)
print(f"\nLignes avant : {len(df_outliers)}")

df_clean_out = df_outliers[df_outliers['age'] >= 0]
print(f"Après age >= 0 : {len(df_clean_out)}")

df_clean_out = df_clean_out[df_clean_out['age'] <= 120]
print(f"Après age <= 120 : {len(df_clean_out)}")

In [ ]:
# Supprimer selon IQR
Q1, Q3 = df_clean_out['salaire'].quantile([0.25, 0.75])
IQR = Q3 - Q1
borne_inf = Q1 - 1.5 * IQR
borne_sup = Q3 + 1.5 * IQR

print(f"\nBornes IQR : [{borne_inf:.0f}, {borne_sup:.0f}]")

df_clean_iqr = df_clean_out[(df_clean_out['salaire'] >= borne_inf) & (df_clean_out['salaire'] <= borne_sup)]
print(f"Lignes après IQR : {len(df_clean_iqr)}")
print(f"Lignes supprimées : {len(df_clean_out) - len(df_clean_iqr)}")

### Winsorisation (capping)

In [ ]:
def winsorize(series, lower_percentile=0.01, upper_percentile=0.99):
    """Winsorise une série aux percentiles spécifiés."""
    lower = series.quantile(lower_percentile)
    upper = series.quantile(upper_percentile)
    return series.clip(lower=lower, upper=upper)

# Appliquer la winsorisation
df_outliers_copy = df_outliers.copy()
df_outliers_copy['salaire_winsorized'] = winsorize(df_outliers_copy['salaire'])

print("Comparaison avant/après winsorisation :")
print(f"Min original : {df_outliers_copy['salaire'].min():.0f}")
print(f"Min winsorisé : {df_outliers_copy['salaire_winsorized'].min():.0f}")
print(f"Max original : {df_outliers_copy['salaire'].max():.0f}")
print(f"Max winsorisé : {df_outliers_copy['salaire_winsorized'].max():.0f}")

### ✍️ Exercice 5.4 : Traitement d'outliers (15 min)

In [ ]:
import pandas as pd
import numpy as np

# Données avec outliers variés
np.random.seed(42)
df_ex4 = pd.DataFrame({
    'employe_id': range(100),
    'age': np.concatenate([np.random.randint(22, 65, 97), [-5, 150, 35]]),
    'salaire': np.concatenate([np.random.normal(50000, 10000, 96), [500000, -1000, 48000, 52000]]),
    'heures_travaillees': np.concatenate([np.random.normal(40, 5, 98), [168, 200]])
})

# Analyse
print("Statistiques descriptives :")
print(df_ex4.describe())

In [ ]:
print(f"\nLignes initiales : {len(df_ex4)}")

# 1. L'âge de -5 : Erreur évidente (âge ne peut pas être négatif)
df_ex4 = df_ex4[df_ex4['age'] >= 0]
print(f"Après suppression age < 0 : {len(df_ex4)} lignes")

# 2. L'âge de 150 : Erreur (impossible d'avoir 150 ans)
df_ex4 = df_ex4[df_ex4['age'] <= 120]
print(f"Après suppression age > 120 : {len(df_ex4)} lignes")

# 3. Le salaire de 500000 : CEO ou erreur ? Sans contexte, on winsorise
df_ex4['salaire'] = df_ex4['salaire'].clip(upper=df_ex4['salaire'].quantile(0.99))
print(f"Salaire max après winsorisation : {df_ex4['salaire'].max():.0f}")

# 4. Le salaire de -1000 : Erreur évidente
df_ex4 = df_ex4[df_ex4['salaire'] >= 0]
print(f"Après suppression salaire < 0 : {len(df_ex4)} lignes")

# 5. Les heures travaillées > 168 (heures dans une semaine)
df_ex4 = df_ex4[df_ex4['heures_travaillees'] <= 168]
print(f"\nLignes finales : {len(df_ex4)}")

> 💭 **Question Socratique #3** : Un data scientist supprime tous les outliers de son dataset avant d'entraîner un modèle de détection de fraude. Voyez-vous le problème avec cette approche ?

**Réponse** : Les fraudes SONT des outliers ! En les supprimant, on enlève exactement ce qu'on cherche à détecter.

---

## 5.4 Nettoyage des types de données

### Conversion des dates

In [ ]:
# Données avec différents formats de dates
df_dates = pd.DataFrame({
    'date_iso': ['2024-01-15', '2024-02-20', '2024-03-25'],
    'date_fr': ['15/01/2024', '20/02/2024', '25/03/2024'],
    'date_us': ['01/15/2024', '02/20/2024', '03/25/2024'],
    'date_erreur': ['15/01/2024', '31/02/2024', '25/03/2024']  # 31 février n'existe pas
})

print("Types avant conversion :")
print(df_dates.dtypes)

In [ ]:
# Conversion basique (format ISO)
df_dates['date_iso_clean'] = pd.to_datetime(df_dates['date_iso'])
print("Format ISO converti :")
print(df_dates['date_iso_clean'])

In [ ]:
# Avec format spécifique (français)
df_dates['date_fr_clean'] = pd.to_datetime(df_dates['date_fr'], format='%d/%m/%Y')
print("\nFormat FR converti :")
print(df_dates['date_fr_clean'])

In [ ]:
# Gestion des erreurs (31 février → NaT)
df_dates['date_erreur_clean'] = pd.to_datetime(df_dates['date_erreur'], format='%d/%m/%Y', errors='coerce')
print("\nFormat avec erreur (31/02 → NaT) :")
print(df_dates['date_erreur_clean'])

### Extraction de composantes temporelles

In [ ]:
# Créer une colonne date
df_temps = pd.DataFrame({
    'date': pd.to_datetime(['2024-01-15', '2024-06-20', '2024-12-25'])
})

# Extraire des composantes temporelles
df_temps['annee'] = df_temps['date'].dt.year
df_temps['mois'] = df_temps['date'].dt.month
df_temps['jour'] = df_temps['date'].dt.day
df_temps['jour_semaine'] = df_temps['date'].dt.dayofweek  # 0=lundi
df_temps['trimestre'] = df_temps['date'].dt.quarter
df_temps['semaine'] = df_temps['date'].dt.isocalendar().week

print("Composantes extraites :")
print(df_temps)

### Conversion des numériques

In [ ]:
# Données avec formats variés
df_num = pd.DataFrame({
    'prix_fr': ['1 234,56 €', '987,00 €', '45,99 €'],
    'taux': ['5,5%', '20%', '10%'],
    'quantite': ['100', '50', 'vingt']  # 'vingt' n'est pas convertible
})

print("Types avant conversion :")
print(df_num.dtypes)

In [ ]:
# Gestion des formats français (virgule décimale, espaces, €)
df_num['prix_clean'] = df_num['prix_fr'].str.replace(' ', '').str.replace('€', '').str.replace(',', '.')
df_num['prix_clean'] = pd.to_numeric(df_num['prix_clean'])
print("Prix nettoyés :")
print(df_num['prix_clean'])

In [ ]:
# Pourcentages en décimales
df_num['taux_clean'] = df_num['taux'].str.replace(',', '.').str.replace('%', '')
df_num['taux_clean'] = pd.to_numeric(df_num['taux_clean']) / 100
print("\nTaux en décimales :")
print(df_num['taux_clean'])

In [ ]:
# Gestion des erreurs de conversion
df_num['quantite_clean'] = pd.to_numeric(df_num['quantite'], errors='coerce')
print("\nQuantités (erreurs → NaN) :")
print(df_num['quantite_clean'])

### ✍️ Exercice 5.5 : Conversion de types (15 min)

In [ ]:
import pandas as pd

# Données avec formats variés
df_ex5 = pd.DataFrame({
    'date_fr': ['15/01/2024', '28/02/2024', '31/04/2024', '10/03/2024'],  # 31 avril n'existe pas
    'prix_fr': ['1 234,56 €', '987,00 €', '45,99 €', '2 500,00 €'],
    'taux': ['5,5%', '20%', '10%', '7,5%'],
    'quantite': ['100', '50', 'vingt', '75']
})

print("Avant conversion :")
print(df_ex5.dtypes)

In [ ]:
# 1. Convertir les dates (gérer l'erreur du 31 avril)
df_ex5['date_clean'] = pd.to_datetime(df_ex5['date_fr'], format='%d/%m/%Y', errors='coerce')

# 2. Convertir les prix
df_ex5['prix_clean'] = df_ex5['prix_fr'].str.replace(' ', '').str.replace('€', '').str.replace(',', '.')
df_ex5['prix_clean'] = pd.to_numeric(df_ex5['prix_clean'])

# 3. Convertir les taux en décimales
df_ex5['taux_clean'] = df_ex5['taux'].str.replace(',', '.').str.replace('%', '')
df_ex5['taux_clean'] = pd.to_numeric(df_ex5['taux_clean']) / 100

# 4. Convertir les quantités (gérer 'vingt')
df_ex5['quantite_clean'] = pd.to_numeric(df_ex5['quantite'], errors='coerce')

print("\nAprès conversion :")
print(df_ex5[['date_clean', 'prix_clean', 'taux_clean', 'quantite_clean']])

print("\nTypes après conversion :")
print(df_ex5[['date_clean', 'prix_clean', 'taux_clean', 'quantite_clean']].dtypes)

---

## 5.5 Nettoyage de texte

### Opérations de base

In [ ]:
# Données textuelles désordonnées
df_texte = pd.DataFrame({
    'nom': ['  Jean DUPONT  ', 'marie-claire Martin', 'PIERRE durand', 'émilie Côté'],
    'email': ['Jean.Dupont@Gmail.COM', 'marie@test.fr', 'PIERRE123@yahoo.fr', 'emilie@test'],
    'telephone': ['06 12 34 56 78', '0687654321', '+33 6 11 22 33 44', '06-99-88-77-66'],
    'adresse': ['12 rue de Paris, 75001 PARIS', '5 avenue Lyon 69001', 'Marseille', '10 bd Bordeaux 33000']
})

print("Avant nettoyage :")
print(df_texte)

In [ ]:
# Supprimer les espaces et normaliser la casse
df_texte['nom_clean'] = df_texte['nom'].str.strip().str.title()
print("Noms nettoyés (strip + title) :")
print(df_texte['nom_clean'])

In [ ]:
# Normaliser les emails (lower, strip)
df_texte['email_clean'] = df_texte['email'].str.lower().str.strip()
print("\nEmails nettoyés :")
print(df_texte['email_clean'])

In [ ]:
# Nettoyer les téléphones (garder uniquement les chiffres)
df_texte['tel_clean'] = df_texte['telephone'].str.replace(r'[^\d]', '', regex=True)
# Normaliser au format français (commencer par 0)
df_texte['tel_clean'] = df_texte['tel_clean'].str.replace('^33', '0', regex=True)

print("\nTéléphones nettoyés :")
print(df_texte['tel_clean'])

### Expressions régulières

In [ ]:
import re

# Extraire le code postal de l'adresse
df_texte['code_postal'] = df_texte['adresse'].str.extract(r'(\d{5})')
print("Codes postaux extraits :")
print(df_texte[['adresse', 'code_postal']])

In [ ]:
# Valider le format email
pattern_email = r'^[\w\.-]+@[\w\.-]+\.\w+$'
df_texte['email_valide'] = df_texte['email_clean'].str.match(pattern_email)
print("\nValidation des emails :")
print(df_texte[['email_clean', 'email_valide']])

### Standardisation des catégories

In [ ]:
# Problème courant : variations d'écriture
df_pays = pd.DataFrame({
    'pays': ['France', 'france', 'FRANCE', 'FR', 'francia', 'Allemagne', 'DE', 'germany']
})

print("Variations de pays :")
print(df_pays['pays'].unique())

In [ ]:
# Solution : mapping
mapping_pays = {
    'france': 'France',
    'fr': 'France',
    'francia': 'France',
    'allemagne': 'Allemagne',
    'de': 'Allemagne',
    'germany': 'Allemagne'
}

df_pays['pays_clean'] = df_pays['pays'].str.lower().map(mapping_pays).fillna(df_pays['pays'])
print("\nPays standardisés :")
print(df_pays)

### ✍️ Exercice 5.6 : Nettoyage de texte complet (20 min)

In [ ]:
import pandas as pd

# Données textuelles désordonnées
df_ex6 = pd.DataFrame({
    'nom': ['  Jean DUPONT  ', 'marie-claire Martin', 'PIERRE durand', 'émilie Côté'],
    'email': ['Jean.Dupont@Gmail.COM', 'marie@test.fr', 'PIERRE123@yahoo.fr', 'emilie@test'],
    'telephone': ['06 12 34 56 78', '0687654321', '+33 6 11 22 33 44', '06-99-88-77-66'],
    'adresse': ['12 rue de Paris, 75001 PARIS', '5 avenue Lyon 69001', 'Marseille', '10 bd Bordeaux 33000']
})

print("Avant nettoyage :")
print(df_ex6)

In [ ]:
# 1. Normaliser les noms (strip, title case)
df_ex6['nom_clean'] = df_ex6['nom'].str.strip().str.title()

# 2. Normaliser les emails (lower, strip)
df_ex6['email_clean'] = df_ex6['email'].str.lower().str.strip()

# 3. Valider les emails (contient @ et .)
df_ex6['email_valide'] = df_ex6['email_clean'].str.contains('@') & df_ex6['email_clean'].str.contains(r'\.')

# 4. Nettoyer les téléphones (garder uniquement les chiffres)
df_ex6['tel_clean'] = df_ex6['telephone'].str.replace(r'[^\d]', '', regex=True)
# Normaliser au format français (commencer par 0)
df_ex6['tel_clean'] = df_ex6['tel_clean'].str.replace('^33', '0', regex=True)

# 5. Extraire le code postal de l'adresse
df_ex6['code_postal'] = df_ex6['adresse'].str.extract(r'(\d{5})')

print("\nAprès nettoyage :")
print(df_ex6[['nom_clean', 'email_clean', 'email_valide', 'tel_clean', 'code_postal']])

---

## 5.6 Traçabilité et documentation

### Pourquoi documenter ?

1. **Reproductibilité** : Pouvoir refaire exactement le même nettoyage
2. **Audit** : Justifier les choix auprès des stakeholders
3. **Collaboration** : Permettre à d'autres de comprendre vos décisions
4. **Debugging** : Identifier d'où vient un problème

In [ ]:
# Template de documentation
import json

# === DOCUMENTATION DU NETTOYAGE ===
nettoyage_log = {
    'date': '2025-01-15',
    'auteur': 'Data Analyst',
    'fichier_source': 'clients_export_2024.csv',
    'lignes_initiales': 0,
    'operations': []
}

print("Template de log créé.")

In [ ]:
# Exemple de nettoyage documenté
df_doc = pd.DataFrame({
    'id': [1, 2, 3, 4, 5, 5, 7, 8],
    'nom': ['Alice', 'Bob', None, 'David', 'Eve', 'Eve', 'Grace', 'Henry'],
    'age': [25, 30, 35, -5, 28, 28, 150, 42],
    'email': ['a@t.com', 'b@t.com', 'c@t.com', 'd@t.com', 'e@t.com', 'e@t.com', 'g@t.com', 'h@t.com']
})

lignes_initiales = len(df_doc)
nettoyage_log['lignes_initiales'] = lignes_initiales

print(f"Lignes initiales : {lignes_initiales}")

In [ ]:
# Étape 1 : Suppression des doublons
doublons_avant = df_doc.duplicated().sum()
df_doc = df_doc.drop_duplicates()
nettoyage_log['operations'].append({
    'etape': 1,
    'operation': 'Suppression doublons exacts',
    'lignes_supprimees': int(doublons_avant),
    'lignes_restantes': len(df_doc)
})
print(f"Étape 1 : Supprimé {doublons_avant} doublons")

In [ ]:
# Étape 2 : Traitement des noms manquants
missing_avant = df_doc['nom'].isnull().sum()
df_doc = df_doc.dropna(subset=['nom'])
nettoyage_log['operations'].append({
    'etape': 2,
    'operation': 'Suppression lignes sans nom',
    'lignes_supprimees': int(missing_avant),
    'justification': 'Nom requis pour identification'
})
print(f"Étape 2 : Supprimé {missing_avant} lignes sans nom")

In [ ]:
# Étape 3 : Correction des âges invalides
ages_invalides = len(df_doc[(df_doc['age'] < 0) | (df_doc['age'] > 120)])
df_doc = df_doc[(df_doc['age'] >= 0) & (df_doc['age'] <= 120)]
nettoyage_log['operations'].append({
    'etape': 3,
    'operation': 'Suppression âges hors plage [0-120]',
    'lignes_supprimees': int(ages_invalides)
})
print(f"Étape 3 : Supprimé {ages_invalides} âges invalides")

In [ ]:
# Résumé
nettoyage_log['lignes_finales'] = len(df_doc)
nettoyage_log['taux_retention'] = f"{len(df_doc)/lignes_initiales*100:.1f}%"

print("\n=== LOG DE NETTOYAGE ===")
print(json.dumps(nettoyage_log, indent=2))

---

## 🧠 Réflexion métacognitive

### Auto-évaluation

| Compétence | 1 | 2 | 3 | 4 | 5 |
|------------|---|---|---|---|---|
| Je sais choisir entre suppression, imputation et flag | ○ | ○ | ○ | ○ | ○ |
| Je peux identifier et supprimer les doublons | ○ | ○ | ○ | ○ | ○ |
| Je sais traiter les outliers selon le contexte | ○ | ○ | ○ | ○ | ○ |
| Je peux convertir les types (dates, numériques) | ○ | ○ | ○ | ○ | ○ |
| Je sais nettoyer du texte (strip, regex) | ○ | ○ | ○ | ○ | ○ |
| Je documente mes choix de nettoyage | ○ | ○ | ○ | ○ | ○ |

### Questions de réflexion

1. **Quelle stratégie de nettoyage** vous semble la plus risquée (celle où on peut perdre de l'information) ?

2. **Comment justifieriez-vous** vos choix de nettoyage à un manager non-technique ?

3. **Quel est le piège principal** à éviter lors du nettoyage de données ?

---

## 📚 Résumé du chapitre

### Points clés à retenir

1. **Valeurs manquantes** :
   - Suppression si < 5% et MCAR
   - Imputation par médiane/moyenne (numérique) ou mode (catégoriel)
   - Flag pour préserver l'information du missing

2. **Doublons** :
   - Normaliser avant de comparer (lower, strip)
   - `drop_duplicates(subset=[...], keep='first'/'last')`

3. **Outliers** :
   - Suppression pour erreurs évidentes
   - Winsorisation pour conserver toutes les lignes
   - **Toujours justifier par le contexte métier**

4. **Types de données** :
   - `pd.to_datetime()` avec `errors='coerce'`
   - `pd.to_numeric()` pour les conversions
   - `str.replace()` et regex pour le texte

5. **Documentation** :
   - Toujours logger chaque opération
   - Justifier les choix
   - Calculer le taux de rétention

---

## ➡️ Prochain chapitre

**Chapitre 6 : Structuration et transformation** — Vous apprendrez à restructurer vos données (pivot, melt), les combiner (merge, concat), et créer de nouvelles variables (feature engineering).

---

*Module 2 — Pipeline Data | Chapitre 5 sur 11*